In [1]:
# Environment and import path setup
import os, sys, site, torch
os.environ['PYTHONNOUSERSITE'] = '1'
usr = site.getusersitepackages()
sys.path = [p for p in sys.path if p != usr]

# Reduce thread contention
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'
torch.set_num_threads(1)

from pathlib import Path

def _add_repo_root_to_sys_path():
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        if (base / 'med3pipe').is_dir():
            if str(base) not in sys.path:
                sys.path.insert(0, str(base))
            print('Added repo root to sys.path:', base)
            return base
    raise RuntimeError("Could not locate 'med3pipe/' in current or parent directories.")

repo_root = _add_repo_root_to_sys_path()

Added repo root to sys.path: C:\Users\cahel\Desktop\Med3Tab-PFN


In [2]:
# Core imports
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torchio as tio
import SimpleITK as sitk
import yaml
import medim
from datetime import datetime
from tqdm import tqdm
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report

from med3pipe.data.prepare import (
    find_case_dirs,
    find_image_files,
    find_segmentation_files,
    merge_images,
    merge_segmentations,
)
from med3pipe.sam.core import find_default_sam3d_root, load_labels_from_sheet, build_y
from med3pipe.sam.transforms import ResizeLargestTo

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

PyTorch version: 2.6.0+cpu
CUDA available: False


## Configuration

In [3]:
# Configuration
config_path = repo_root / 'configs' / 'datasets_analysis.yaml'
results_dir = repo_root / 'results'
results_dir.mkdir(parents=True, exist_ok=True)

# SAM-Med3D settings
sam3d_root = find_default_sam3d_root()
checkpoint_path = sam3d_root / 'ckpt' / 'sam_med3d_turbo.pth'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
img_size = 128

# ROI cropping settings
ROI_MARGIN = 30

# Datasets to evaluate
DATASETS_TO_RUN = ['gist', 'lipo']

# TabPFN settings
N_SPLITS = 5  # K-fold cross-validation
N_COMPONENTS_MAX = 500  # Max PCA components
RANDOM_STATE = 42

print(f"Config path: {config_path}")
print(f"Config exists: {config_path.exists()}")
print(f"Checkpoint path: {checkpoint_path}")
print(f"Checkpoint exists: {checkpoint_path.exists()}")
print(f"Device: {device}")
print(f"Results dir: {results_dir}")

Config path: C:\Users\cahel\Desktop\Med3Tab-PFN\configs\datasets_analysis.yaml
Config exists: True
Checkpoint path: C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\ckpt\sam_med3d_turbo.pth
Checkpoint exists: True
Device: cpu
Results dir: C:\Users\cahel\Desktop\Med3Tab-PFN\results


## Load SAM-Med3D Model via medim

In [4]:
# Load model via medim
print("Loading SAM-Med3D model via medim...")
model = medim.create_model(
    "SAM-Med3D",
    pretrained=True,
    checkpoint_path=str(checkpoint_path)
).to(device)
model.eval()
print("✅ SAM-Med3D model loaded successfully!")

Loading SAM-Med3D model via medim...
creating model SAM-Med3D
try to load pretrained weights from C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\ckpt\sam_med3d_turbo.pth
✅ SAM-Med3D model loaded successfully!


## ROI Cropping Helper Functions

In [5]:
def _znorm_masking_method(x):
    """Masking method for ZNormalization."""
    return x > 0


def get_bounding_box(mask_array):
    """Get bounding box of non-zero region in 3D mask."""
    where = np.where(mask_array > 0)
    if len(where[0]) == 0:
        return None
    z_min, z_max = where[0].min(), where[0].max()
    y_min, y_max = where[1].min(), where[1].max()
    x_min, x_max = where[2].min(), where[2].max()
    return (z_min, z_max, y_min, y_max, x_min, x_max)


def add_margin_to_bbox(bbox, margin, shape):
    """Add margin to bounding box, respecting image boundaries."""
    z_min, z_max, y_min, y_max, x_min, x_max = bbox
    z_min = max(0, z_min - margin)
    z_max = min(shape[0] - 1, z_max + margin)
    y_min = max(0, y_min - margin)
    y_max = min(shape[1] - 1, y_max + margin)
    x_min = max(0, x_min - margin)
    x_max = min(shape[2] - 1, x_max + margin)
    return (z_min, z_max, y_min, y_max, x_min, x_max)


def crop_array_to_roi(array, bbox):
    """Crop array to ROI."""
    z_min, z_max, y_min, y_max, x_min, x_max = bbox
    return array[z_min:z_max+1, y_min:y_max+1, x_min:x_max+1]


def load_volume_roi_for_sam(img_source, mask_source, img_size=128, roi_margin=30, modality='CT'):
    """
    Load volume with ROI-centric preprocessing for SAM.
    Returns image tensor (1, 1, D, H, W) and mask array (D, H, W).
    """
    # Load image
    if isinstance(img_source, sitk.Image):
        sitk_img = img_source
    else:
        sitk_img = sitk.ReadImage(str(img_source))
    sitk_arr_img, _ = tio.data.io.sitk_to_nib(sitk_img)
    
    # Load mask
    if isinstance(mask_source, sitk.Image):
        sitk_mask = mask_source
    else:
        sitk_mask = sitk.ReadImage(str(mask_source))
    sitk_arr_mask, _ = tio.data.io.sitk_to_nib(sitk_mask)
    
    # Get numpy arrays
    img_np = sitk_arr_img.squeeze()
    mask_np = sitk_arr_mask.squeeze()
    
    # Get bounding box from mask
    bbox = get_bounding_box(mask_np)
    if bbox is None:
        bbox = (0, mask_np.shape[0]-1, 0, mask_np.shape[1]-1, 0, mask_np.shape[2]-1)
    
    # Add margin
    bbox = add_margin_to_bbox(bbox, roi_margin, mask_np.shape)
    
    # Crop to ROI
    img_cropped = crop_array_to_roi(img_np, bbox)
    mask_cropped = crop_array_to_roi(mask_np, bbox)
    
    # Create TorchIO subjects from cropped arrays
    img_tensor = torch.from_numpy(img_cropped).float().unsqueeze(0)
    mask_tensor = torch.from_numpy(mask_cropped).float().unsqueeze(0)
    
    subject_img = tio.Subject(image=tio.ScalarImage(tensor=img_tensor))
    subject_mask = tio.Subject(label=tio.LabelMap(tensor=mask_tensor))
    
    # Apply CT clamping if needed
    if modality.upper() == 'CT':
        subject_img = tio.Clamp(-1000, 1000)(subject_img)
    
    # Transform for SAM: Resize + Pad + ZNormalization
    transform = tio.Compose([
        tio.ToCanonical(),
        ResizeLargestTo(target_size=img_size),
        tio.CropOrPad(target_shape=(img_size, img_size, img_size)),
        tio.ZNormalization(masking_method=_znorm_masking_method),
    ])
    
    mask_transform = tio.Compose([
        tio.ToCanonical(),
        ResizeLargestTo(target_size=img_size),
        tio.CropOrPad(target_shape=(img_size, img_size, img_size)),
    ])
    
    subject_img = transform(subject_img)
    subject_mask = mask_transform(subject_mask)
    
    image = subject_img.image.data.clone().detach().unsqueeze(0).float()  # (1, 1, D, H, W)
    mask = subject_mask.label.data.squeeze().numpy()
    mask = (mask > 0).astype(float)
    
    return image, mask


print("✅ ROI cropping helper functions defined.")

✅ ROI cropping helper functions defined.


## Pooling Strategy Implementations

In [6]:
def pool_avg(embedding):
    """
    Average Pooling: Global average across all spatial dimensions.
    
    Args:
        embedding: (1, C, D, H, W) tensor
    
    Returns:
        (C,) numpy array
    """
    # Global average pooling
    pooled = embedding.mean(dim=(2, 3, 4))  # (1, C)
    return pooled.squeeze(0).cpu().numpy()


def pool_multiscale(embedding):
    """
    Multiscale Pooling: Concatenation of features at multiple spatial resolutions.
    
    Creates features at 3 scales:
    - 1x1x1 (global average)
    - 2x2x2 (8 local regions)
    - 4x4x4 (64 local regions)
    
    Args:
        embedding: (1, C, D, H, W) tensor
    
    Returns:
        (C * (1 + 8 + 64),) = (C * 73,) numpy array
    """
    B, C, D, H, W = embedding.shape
    features = []
    
    # Scale 1: 1x1x1 (global average)
    f1 = F.adaptive_avg_pool3d(embedding, (1, 1, 1))  # (1, C, 1, 1, 1)
    features.append(f1.view(B, -1))  # (1, C)
    
    # Scale 2: 2x2x2 (8 local regions)
    f2 = F.adaptive_avg_pool3d(embedding, (2, 2, 2))  # (1, C, 2, 2, 2)
    features.append(f2.view(B, -1))  # (1, C*8)
    
    # Scale 3: 4x4x4 (64 local regions)
    f4 = F.adaptive_avg_pool3d(embedding, (4, 4, 4))  # (1, C, 4, 4, 4)
    features.append(f4.view(B, -1))  # (1, C*64)
    
    # Concatenate all scales
    pooled = torch.cat(features, dim=1)  # (1, C*73)
    return pooled.squeeze(0).cpu().numpy()


def pool_percentile(embedding):
    """
    Percentile Pooling: Captures the distribution of feature activations.
    
    Computes multiple percentiles per channel:
    - 10th, 25th, 50th (median), 75th, 90th percentiles
    
    Args:
        embedding: (1, C, D, H, W) tensor
    
    Returns:
        (C * 5,) numpy array (5 percentiles per channel)
    """
    B, C, D, H, W = embedding.shape
    
    # Flatten spatial dimensions: (B, C, D*H*W)
    flat = embedding.view(B, C, -1)  # (1, C, N) where N = D*H*W
    
    # Compute percentiles for each channel
    percentiles = [10, 25, 50, 75, 90]
    features = []
    
    for p in percentiles:
        # torch.quantile expects values between 0 and 1
        q = p / 100.0
        perc = torch.quantile(flat, q, dim=2)  # (1, C)
        features.append(perc)
    
    # Concatenate all percentiles: (1, C*5)
    pooled = torch.cat(features, dim=1)
    return pooled.squeeze(0).cpu().numpy()


# Define pooling strategies
POOLING_STRATEGIES = {
    'avg': pool_avg,
    'multiscale': pool_multiscale,
    'percentile': pool_percentile,
}

print("✅ Pooling strategies defined:")
for name in POOLING_STRATEGIES:
    print(f"   - {name}")

✅ Pooling strategies defined:
   - avg
   - multiscale
   - percentile


## Dataset Discovery

In [32]:
def infer_case_id(case_dir):
    """Infer case identifier from a NIFTI directory path."""
    case_dir = Path(case_dir).resolve()
    if case_dir.name.lower() == 'nifti':
        parent = case_dir.parent
        grandparent = parent.parent if parent else None
        if grandparent and grandparent.name:
            return grandparent.name
        if parent.name:
            return parent.name
    return case_dir.name


def discover_cases_from_config(config_path, project_root, dataset_filter=None):
    """Discover all dataset cases from config file."""
    with open(config_path) as f:
        config = yaml.safe_load(f)
    
    datasets_cfg = config.get('datasets', {})
    discovered = {}
    
    for dataset_name, cfg in datasets_cfg.items():
        # Apply filter if specified
        if dataset_filter and dataset_name not in dataset_filter:
            continue
        
        dataset_root = Path(cfg.get('dataset_root', '')).expanduser()
        if not dataset_root.is_absolute():
            dataset_root = (project_root / dataset_root).resolve()
        else:
            dataset_root = dataset_root.resolve()
        
        if not dataset_root.exists():
            print(f"⚠️  Dataset root missing for {dataset_name}: {dataset_root}")
            continue
        
        case_glob = cfg.get('prepare', {}).get('case_glob')
        case_dirs = find_case_dirs(dataset_root, case_glob=case_glob)
        
        cases = []
        for case_dir in case_dirs:
            image_files = find_image_files(case_dir)
            seg_files = find_segmentation_files(case_dir)
            if not image_files or not seg_files:
                continue
            case_id = infer_case_id(case_dir)
            cases.append({
                'case_id': case_id,
                'image_files': image_files,
                'segmentation_files': seg_files,
            })
        
        if not cases:
            print(f"⚠️  No valid cases found for {dataset_name}")
            continue
        
        # Get label info
        labels_cfg = cfg.get('labels', {})
        sheet_csv = labels_cfg.get('sheet_csv', 'sheet.csv')
        sheet_path = Path(sheet_csv) if Path(sheet_csv).is_absolute() else dataset_root / sheet_csv
        
        # Fallback: if sheet not found in dataset dir, try data/ parent dir
        if not sheet_path.exists():
            data_dir = dataset_root.parent  # Assume dataset_root is data/gist, so parent is data/
            fallback_path = data_dir / 'sheet.csv'
            if fallback_path.exists():
                sheet_path = fallback_path
        
        discovered[dataset_name] = {
            'dataset': dataset_name,
            'category': cfg.get('category', dataset_name),
            'modality': cfg.get('modality', 'CT'),
            'cases': cases,
            'sheet_csv': sheet_path,
            'dataset_name_in_sheet': labels_cfg.get('dataset_name'),
            'subject_col': labels_cfg.get('subject_col', 'Subject'),
            'label_col': labels_cfg.get('label_col', 'Diagnosis_binary'),
            'case_suffix': labels_cfg.get('case_suffix', '_CT'),
        }
    
    return discovered


# Discover datasets
datasets = discover_cases_from_config(config_path, repo_root, dataset_filter=DATASETS_TO_RUN)

print(f"\n✅ Discovered {len(datasets)} datasets:")
for name, info in datasets.items():
    print(f"   - {name}: {len(info['cases'])} cases")


✅ Discovered 2 datasets:
   - gist: 246 cases
   - lipo: 115 cases


## Extract Embeddings with ROI Cropping

In [8]:
@torch.no_grad()
def extract_embeddings_for_dataset(model, dataset_info, device, img_size=128, roi_margin=30):
    """
    Extract SAM-Med3D embeddings for all cases in a dataset.
    
    Returns:
        embeddings: list of (1, C, D, H, W) tensors
        case_ids: list of case ID strings
    """
    model.eval()
    embeddings = []
    case_ids = []
    modality = dataset_info.get('modality', 'CT')
    
    for case in tqdm(dataset_info['cases'], desc=f"Extracting {dataset_info['dataset']}"):
        case_id = case['case_id']
        try:
            # Merge images and segmentations
            image_sitk = merge_images(case['image_files'])
            mask_sitk = merge_segmentations(case['segmentation_files'])
            
            # Load with ROI cropping
            image_tensor, _ = load_volume_roi_for_sam(
                image_sitk, mask_sitk,
                img_size=img_size,
                roi_margin=roi_margin,
                modality=modality,
            )
            
            # Extract embedding via image encoder
            image_tensor = image_tensor.to(device)
            embedding = model.image_encoder(image_tensor)  # (1, C, d, h, w)
            
            embeddings.append(embedding.cpu())
            case_ids.append(case_id)
            
        except Exception as e:
            print(f"   ⚠️ Error processing {case_id}: {e}")
            continue
    
    return embeddings, case_ids


# Extract embeddings for all datasets
all_embeddings = {}
all_case_ids = {}

for ds_name, ds_info in datasets.items():
    print(f"\n{'='*60}")
    print(f"Extracting embeddings for: {ds_name}")
    print(f"{'='*60}")
    
    embeddings, case_ids = extract_embeddings_for_dataset(
        model, ds_info, device,
        img_size=img_size, roi_margin=ROI_MARGIN
    )
    
    all_embeddings[ds_name] = embeddings
    all_case_ids[ds_name] = case_ids
    
    print(f"✅ Extracted {len(embeddings)} embeddings")
    if embeddings:
        print(f"   Embedding shape: {embeddings[0].shape}")


Extracting embeddings for: gist


Extracting gist:   4%|▎         | 9/246 [03:28<2:05:25, 31.75s/it]c:\Users\cahel\.conda\envs\sammed3d\lib\site-packages\torchio\transforms\transform.py:158: RuntimeWarning: Output shape (128, 85, 49) != target shape (np.int64(128), np.int64(85), np.int64(50)). Fixing with CropOrPad
  transformed = self.apply_transform(subject)
Extracting gist:   7%|▋         | 18/246 [06:00<1:40:45, 26.52s/it]c:\Users\cahel\.conda\envs\sammed3d\lib\site-packages\torchio\transforms\transform.py:158: RuntimeWarning: Output shape (99, 128, 49) != target shape (np.int64(100), np.int64(128), np.int64(49)). Fixing with CropOrPad
  transformed = self.apply_transform(subject)
Extracting gist:   9%|▉         | 23/246 [08:10<1:15:57, 20.44s/it]c:\Users\cahel\.conda\envs\sammed3d\lib\site-packages\torchio\transforms\transform.py:158: RuntimeWarning: Output shape (128, 124, 55) != target shape (np.int64(128), np.int64(124), np.int64(56)). Fixing with CropOrPad
  transformed = self.apply_transform(subject)
Extracti

✅ Extracted 246 embeddings
   Embedding shape: torch.Size([1, 384, 8, 8, 8])

Extracting embeddings for: lipo


Extracting lipo:   3%|▎         | 3/115 [00:23<14:35,  7.82s/it]c:\Users\cahel\.conda\envs\sammed3d\lib\site-packages\torchio\transforms\transform.py:158: RuntimeWarning: Output shape (54, 127, 5) != target shape (np.int64(55), np.int64(127), np.int64(5)). Fixing with CropOrPad
  transformed = self.apply_transform(subject)
Extracting lipo:  22%|██▏       | 25/115 [03:12<11:32,  7.70s/it]c:\Users\cahel\.conda\envs\sammed3d\lib\site-packages\torchio\transforms\transform.py:158: RuntimeWarning: Output shape (125, 128, 21) != target shape (np.int64(125), np.int64(128), np.int64(22)). Fixing with CropOrPad
  transformed = self.apply_transform(subject)
Extracting lipo:  23%|██▎       | 27/115 [03:28<11:14,  7.66s/it]c:\Users\cahel\.conda\envs\sammed3d\lib\site-packages\torchio\transforms\transform.py:158: RuntimeWarning: Output shape (120, 128, 25) != target shape (np.int64(120), np.int64(128), np.int64(26)). Fixing with CropOrPad
  transformed = self.apply_transform(subject)
Extracting lipo

✅ Extracted 115 embeddings
   Embedding shape: torch.Size([1, 384, 8, 8, 8])


## Apply Pooling Strategies

In [9]:
def apply_pooling(embeddings, pooling_fn):
    """
    Apply a pooling function to a list of embeddings.
    
    Returns:
        X: (N, D) numpy array where D depends on the pooling strategy
    """
    features = []
    for emb in embeddings:
        feat = pooling_fn(emb)
        features.append(feat)
    return np.stack(features)


# Create pooled features for each strategy and dataset
pooled_features = {}

for pool_name, pool_fn in POOLING_STRATEGIES.items():
    pooled_features[pool_name] = {}
    print(f"\n{'='*60}")
    print(f"Applying {pool_name} pooling...")
    print(f"{'='*60}")
    
    for ds_name in datasets.keys():
        embeddings = all_embeddings[ds_name]
        if not embeddings:
            continue
        
        X = apply_pooling(embeddings, pool_fn)
        pooled_features[pool_name][ds_name] = X
        print(f"   {ds_name}: {X.shape}")


Applying avg pooling...
   gist: (246, 384)
   lipo: (115, 384)

Applying multiscale pooling...
   gist: (246, 28032)
   lipo: (115, 28032)

Applying percentile pooling...
   gist: (246, 1920)
   lipo: (115, 1920)


## Load Labels

In [33]:
def load_labels_for_dataset(ds_info, case_ids):
    """
    Load labels for the given case IDs.
    
    Returns:
        y: numpy array of labels
        valid_indices: indices of cases with valid labels
    """
    sheet_csv = ds_info['sheet_csv']
    if not sheet_csv.exists():
        print(f"   ⚠️ Sheet not found: {sheet_csv}")
        return None, None
    
    # Pass case_suffix to load_labels_from_sheet so it builds the correct lab_map keys
    case_suffix = ds_info.get('case_suffix', '_CT')
    
    df, lab_map = load_labels_from_sheet(
        sheet_csv,
        dataset_name=ds_info['dataset_name_in_sheet'],
        subject_col=ds_info['subject_col'],
        label_col=ds_info['label_col'],
        case_suffix=case_suffix,  # Pass the suffix here!
    )
    
    # Now try to match with the case_ids as-is (they already have the suffix)
    y, missing = build_y(case_ids, lab_map)
    
    # Find valid indices (non-NaN labels)
    if len(y) == 0:
        return None, None
    
    valid_mask = ~np.isnan(y)
    valid_indices = np.where(valid_mask)[0]
    y_valid = y[valid_mask].astype(int)
    
    if len(missing) > 0:
        print(f"   ⚠️ Missing labels for {len(missing)} cases")
    
    return y_valid, valid_indices


# Load labels for all datasets
all_labels = {}
all_valid_indices = {}

for ds_name, ds_info in datasets.items():
    print(f"\nLoading labels for: {ds_name}")
    case_ids = all_case_ids[ds_name]
    y, valid_idx = load_labels_for_dataset(ds_info, case_ids)
    
    if y is not None:
        all_labels[ds_name] = y
        all_valid_indices[ds_name] = valid_idx
        print(f"   ✅ Loaded {len(y)} labels (class distribution: {np.bincount(y)})")


Loading labels for: gist
   ✅ Loaded 246 labels (class distribution: [121 125])

Loading labels for: lipo
   ✅ Loaded 115 labels (class distribution: [58 57])


## TabPFN Pipeline

In [34]:
# Import TabPFN
from med3pipe.tabular.tabpfn import ensure_tabpfn_on_sys_path

# Ensure TabPFN is available
tabpfn_src = ensure_tabpfn_on_sys_path()
from tabpfn.classifier import TabPFNClassifier

print(f"✅ TabPFN loaded from: {tabpfn_src}")

✅ TabPFN loaded from: None


In [35]:
def run_tabpfn_kfold(X, y, n_splits=5, n_components_max=500, random_state=42, device='cpu'):
    """
    Run TabPFN with k-fold cross-validation.
    
    Returns:
        dict with aggregated metrics
    """
    # Standardize and apply PCA
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    n_components = min(n_components_max, X_scaled.shape[1], X_scaled.shape[0] - 1)
    if n_components < X_scaled.shape[1]:
        pca = PCA(n_components=n_components, random_state=random_state)
        X_pca = pca.fit_transform(X_scaled)
    else:
        X_pca = X_scaled
    
    # K-fold cross-validation
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    
    all_accs = []
    all_f1s = []
    all_aucs = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_pca, y)):
        X_train, X_val = X_pca[train_idx], X_pca[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]
        
        # Train TabPFN
        clf = TabPFNClassifier(device=device)
        clf.fit(X_train, y_train)
        
        # Predict
        y_pred = clf.predict(X_val)
        
        # Metrics
        acc = accuracy_score(y_val, y_pred)
        f1 = f1_score(y_val, y_pred, average='macro')
        
        try:
            proba = clf.predict_proba(X_val)
            if proba.ndim == 1:
                proba = np.stack([1 - proba, proba], axis=-1)
            auc = roc_auc_score(y_val, proba[:, 1])
        except:
            auc = None
        
        all_accs.append(acc)
        all_f1s.append(f1)
        if auc is not None:
            all_aucs.append(auc)
    
    return {
        'accuracy': np.mean(all_accs),
        'accuracy_std': np.std(all_accs),
        'macro_f1': np.mean(all_f1s),
        'macro_f1_std': np.std(all_f1s),
        'roc_auc': np.mean(all_aucs) if all_aucs else None,
        'roc_auc_std': np.std(all_aucs) if all_aucs else None,
        'n_samples': len(y),
        'n_features_original': X.shape[1],
        'n_features_pca': X_pca.shape[1],
    }


print("✅ TabPFN k-fold pipeline defined.")

✅ TabPFN k-fold pipeline defined.


## Run Experiments

In [ ]:
# Run experiments for each pooling strategy and dataset
results = []

for pool_name in POOLING_STRATEGIES.keys():
    print(f"\n{'='*80}")
    print(f"POOLING STRATEGY: {pool_name.upper()}")
    print(f"{'='*80}")
    
    for ds_name in datasets.keys():
        if ds_name not in all_labels:
            print(f"   ⚠️ Skipping {ds_name} (no labels)")
            continue
        
        # Get features and labels
        X_full = pooled_features[pool_name][ds_name]
        valid_idx = all_valid_indices[ds_name]
        y = all_labels[ds_name]
        
        # Filter to valid cases
        X = X_full[valid_idx]
        
        print(f"\n   Dataset: {ds_name}")
        print(f"   Features shape: {X.shape}")
        print(f"   Labels: {len(y)} (class distribution: {np.bincount(y)})")
        
        # Skip if too few samples
        if len(y) < N_SPLITS * 2:
            print(f"   ⚠️ Too few samples for {N_SPLITS}-fold CV")
            continue
        
        # Run TabPFN
        try:
            device_str = 'cuda' if torch.cuda.is_available() else 'cpu'
            metrics = run_tabpfn_kfold(
                X, y,
                n_splits=N_SPLITS,
                n_components_max=N_COMPONENTS_MAX,
                random_state=RANDOM_STATE,
                device=device_str,
            )
            
            print(f"   Results:")
            print(f"      Accuracy: {metrics['accuracy']:.4f} ± {metrics['accuracy_std']:.4f}")
            print(f"      Macro F1: {metrics['macro_f1']:.4f} ± {metrics['macro_f1_std']:.4f}")
            if metrics['roc_auc'] is not None:
                print(f"      ROC AUC:  {metrics['roc_auc']:.4f} ± {metrics['roc_auc_std']:.4f}")
            
            results.append({
                'dataset': ds_name,
                'pooling_strategy': pool_name,
                'accuracy': metrics['accuracy'],
                'accuracy_std': metrics['accuracy_std'],
                'macro_f1': metrics['macro_f1'],
                'macro_f1_std': metrics['macro_f1_std'],
                'roc_auc': metrics['roc_auc'],
                'roc_auc_std': metrics['roc_auc_std'],
                'n_samples': metrics['n_samples'],
                'n_features_original': metrics['n_features_original'],
                'n_features_pca': metrics['n_features_pca'],
            })
            
        except Exception as e:
            print(f"   ❌ Error: {e}")
            results.append({
                'dataset': ds_name,
                'pooling_strategy': pool_name,
                'error': str(e),
            })


POOLING STRATEGY: AVG

   Dataset: gist
   Features shape: (246, 384)
   Labels: 246 (class distribution: [121 125])
   Results:
      Accuracy: 0.6300 ± 0.0176
      Macro F1: 0.6288 ± 0.0172
      ROC AUC:  0.6280 ± 0.0294

   Dataset: lipo
   Features shape: (115, 384)
   Labels: 115 (class distribution: [58 57])
   Results:
      Accuracy: 0.6300 ± 0.0176
      Macro F1: 0.6288 ± 0.0172
      ROC AUC:  0.6280 ± 0.0294

   Dataset: lipo
   Features shape: (115, 384)
   Labels: 115 (class distribution: [58 57])
   Results:
      Accuracy: 0.5913 ± 0.1496
      Macro F1: 0.5891 ± 0.1508
      ROC AUC:  0.6606 ± 0.1256

POOLING STRATEGY: MULTISCALE

   Dataset: gist
   Features shape: (246, 28032)
   Labels: 246 (class distribution: [121 125])
   Results:
      Accuracy: 0.5913 ± 0.1496
      Macro F1: 0.5891 ± 0.1508
      ROC AUC:  0.6606 ± 0.1256

POOLING STRATEGY: MULTISCALE

   Dataset: gist
   Features shape: (246, 28032)
   Labels: 246 (class distribution: [121 125])


## Results Summary

In [14]:
# Create results DataFrame
df_results = pd.DataFrame(results)

print("\n" + "="*80)
print("RESULTS SUMMARY")
print("="*80)
display(df_results)


RESULTS SUMMARY


""


In [15]:
# Create a pivot table for easier comparison
if 'accuracy' in df_results.columns and df_results['accuracy'].notna().any():
    print("\n" + "="*80)
    print("ACCURACY COMPARISON (by pooling strategy and dataset)")
    print("="*80)
    
    pivot_acc = df_results.pivot(
        index='dataset',
        columns='pooling_strategy',
        values='accuracy'
    )
    display(pivot_acc)
    
    print("\n" + "="*80)
    print("ROC AUC COMPARISON (by pooling strategy and dataset)")
    print("="*80)
    
    pivot_auc = df_results.pivot(
        index='dataset',
        columns='pooling_strategy',
        values='roc_auc'
    )
    display(pivot_auc)

In [16]:
# Save results
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_csv = results_dir / f"pooling_comparison_{timestamp}.csv"
df_results.to_csv(output_csv, index=False)
print(f"\n✅ Results saved to: {output_csv}")

# Also save a copy with a fixed name for easy access
output_csv_latest = results_dir / "pooling_comparison_latest.csv"
df_results.to_csv(output_csv_latest, index=False)
print(f"✅ Latest results saved to: {output_csv_latest}")


✅ Results saved to: C:\Users\cahel\Desktop\Med3Tab-PFN\results\pooling_comparison_20251216_163552.csv
✅ Latest results saved to: C:\Users\cahel\Desktop\Med3Tab-PFN\results\pooling_comparison_latest.csv


## Visualization

In [17]:
import matplotlib.pyplot as plt

# Create bar plot comparing pooling strategies
if 'accuracy' in df_results.columns and df_results['accuracy'].notna().any():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Accuracy plot
    ax = axes[0]
    for i, ds_name in enumerate(df_results['dataset'].unique()):
        ds_data = df_results[df_results['dataset'] == ds_name]
        x = np.arange(len(POOLING_STRATEGIES))
        width = 0.35
        offset = (i - 0.5) * width
        
        accs = [ds_data[ds_data['pooling_strategy'] == p]['accuracy'].values[0] 
                if len(ds_data[ds_data['pooling_strategy'] == p]) > 0 else 0 
                for p in POOLING_STRATEGIES.keys()]
        stds = [ds_data[ds_data['pooling_strategy'] == p]['accuracy_std'].values[0] 
                if len(ds_data[ds_data['pooling_strategy'] == p]) > 0 else 0 
                for p in POOLING_STRATEGIES.keys()]
        
        ax.bar(x + offset, accs, width, yerr=stds, label=ds_name, capsize=3)
    
    ax.set_xlabel('Pooling Strategy')
    ax.set_ylabel('Accuracy')
    ax.set_title('Accuracy by Pooling Strategy')
    ax.set_xticks(x)
    ax.set_xticklabels(list(POOLING_STRATEGIES.keys()))
    ax.legend()
    ax.set_ylim(0, 1)
    
    # ROC AUC plot
    ax = axes[1]
    for i, ds_name in enumerate(df_results['dataset'].unique()):
        ds_data = df_results[df_results['dataset'] == ds_name]
        x = np.arange(len(POOLING_STRATEGIES))
        width = 0.35
        offset = (i - 0.5) * width
        
        aucs = [ds_data[ds_data['pooling_strategy'] == p]['roc_auc'].values[0] 
                if len(ds_data[ds_data['pooling_strategy'] == p]) > 0 and 
                   pd.notna(ds_data[ds_data['pooling_strategy'] == p]['roc_auc'].values[0])
                else 0 
                for p in POOLING_STRATEGIES.keys()]
        stds = [ds_data[ds_data['pooling_strategy'] == p]['roc_auc_std'].values[0] 
                if len(ds_data[ds_data['pooling_strategy'] == p]) > 0 and
                   pd.notna(ds_data[ds_data['pooling_strategy'] == p]['roc_auc_std'].values[0])
                else 0 
                for p in POOLING_STRATEGIES.keys()]
        
        ax.bar(x + offset, aucs, width, yerr=stds, label=ds_name, capsize=3)
    
    ax.set_xlabel('Pooling Strategy')
    ax.set_ylabel('ROC AUC')
    ax.set_title('ROC AUC by Pooling Strategy')
    ax.set_xticks(x)
    ax.set_xticklabels(list(POOLING_STRATEGIES.keys()))
    ax.legend()
    ax.set_ylim(0, 1)
    
    plt.tight_layout()
    
    # Save figure
    fig_path = results_dir / f"pooling_comparison_{timestamp}.png"
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f"\n✅ Figure saved to: {fig_path}")
    
    plt.show()

## Summary

This notebook compared three pooling strategies for SAM-Med3D embeddings:

| Strategy | Description | Feature Dim (for C=384) |
|----------|-------------|-------------------------|
| **avg** | Global average pooling | C = 384 |
| **multiscale** | 1x1x1 + 2x2x2 + 4x4x4 pyramid | C * 73 = 28,032 |
| **percentile** | 5 percentiles (10, 25, 50, 75, 90) per channel | C * 5 = 1,920 |

The results are saved to the `/results` folder for further analysis.